# 🎮 Free Minecraft Server on Kaggle

Automatically set up a Minecraft server using Kaggle's free compute resources.

**Important limitations:**
- Kaggle notebooks run for max ~9 hours per session
- NO permanent storage — world data is lost when session ends
- Internet access may be restricted depending on your Kaggle account settings
- No inbound connections — need a tunnel for friends to join
- RAM/CPU limited by your Kaggle tier (typically 2 cores, 13-16 GB RAM free tier)

In [ ]:
# Cell 1: Environment Diagnostics
import platform
import subprocess
import shutil
import os
import sys

print("=" * 60)
print("MINECRAFT SERVER ENVIRONMENT DIAGNOSTICS")
print("=" * 60)

# OS Info
print(f"\n🖥️  Operating System: {platform.system()} {platform.release()}")
print(f"📱 Architecture: {platform.machine()}")
print(f"🐍 Python Version: {platform.python_version()}")

# CPU Cores
try:
    cpu_count = os.cpu_count()
    print(f"⚡ CPU Cores: {cpu_count or 'Unknown'}")
except:
    print("⚠️  Could not detect CPU count")

# RAM (in GB)
try:
    with open('/proc/meminfo', 'r') as f:
        mem_info = f.read()
    total_kb = int([line for line in mem_info.split('\n') if line.startswith('MemTotal:')][0].split()[1])
    total_gb = total_kb / (1024**2)
    print(f"💾 Total RAM: {total_gb:.1f} GB")
except Exception as e:
    print(f"⚠️  RAM detection failed: {e}")

# Disk Space
try:
    total, used, free = shutil.disk_usage('/')
    print(f"💽 Disk - Total: {total/(1024**3):.1f}GB, Free: {free/(1024**3):.1f}GB")
except Exception as e:
    print(f"⚠️  Disk detection failed: {e}")

# Check Java
print("\n🔍 Checking Java...")
java_result = subprocess.run(['java', '-version'], capture_output=True, text=True)
if java_result.returncode == 0:
    print(f"✅ Java found: {java_result.stderr.split('\n')[0] if java_result.stderr else 'Unknown version'}")
else:
    print("❌ Java NOT found - this is required for Minecraft server")

# Internet Test
print("\n🌐 Testing Internet Connectivity...")
try:
    import urllib.request
    urllib.request.urlopen('https://www.minecraft.net', timeout=10)
    print("✅ Internet access OK")
except Exception as e:
    print(f"❌ Internet access issue: {e}")

print("\n" + "=" * 60)
print("NOTE: Kaggle sessions are TEMPORARY. World data is lost when")
print("the notebook stops. Use Google Drive backup to persist your world!")
print("=" * 60)

In [ ]:
# Cell 2: Configuration and Dependency Installation
import os
import subprocess

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)

# ===== EDIT THESE SETTINGS =====
SERVER_EDITION = "java"  # "bedrock" or "java"
SERVER_VERSION = "1.21.4"  # Minecraft version
EULA_ACCEPTED = True  # Set to True after reading https://minecraft.net/eula
VIEW_DISTANCE = 6  # Lower = less lag, higher = see farther
SIMULATION_DISTANCE = 4  # Mob tick distance
MAX_PLAYERS = 5
DIFFICULTY = "normal"  # peaceful, easy, normal, hard
SEED = ""  # Leave empty for random, or set a specific seed
MOTD = "Kaggle Minecraft Server"
ONLINE_MODE = True  # Set False for offline/cracked clients

# Memory allocation (in MB) - be conservative!
# For Kaggle free tier, use 1024-2048MB max
JAVA_XMS = 1024  # Initial heap
JAVA_XMX = 2048  # Max heap (DO NOT exceed available RAM)

# Google Drive Backup Settings
ENABLE_DRIVE_BACKUP = True  # Set to True to enable Google Drive persistence
BACKUP_INTERVAL_MINUTES = 30  # How often to backup during server runtime
MAX_BACKUPS_TO_KEEP = 5  # Number of recent backups to retain
DRY_RUN = False  # Set True to test without actual Drive operations

print(f"📋 Server Edition: {SERVER_EDITION}")
print(f"📋 View Distance: {VIEW_DISTANCE}")
print(f"📋 Max Players: {MAX_PLAYERS}")
print(f"📋 Difficulty: {DIFFICULTY}")
print(f"📋 Seed: {SEED or '(random)'}")
print(f"📋 EULA: {'Accepted' if EULA_ACCEPTED else 'NOT Accepted - server will not start'}")
print(f"📋 Google Drive Backup: {'Enabled' if ENABLE_DRIVE_BACKUP else 'Disabled'}")

# Create server directory
SERVER_DIR = "/kaggle/working/minecraft_server"
os.makedirs(SERVER_DIR, exist_ok=True)
print(f"\n📁 Server directory: {SERVER_DIR}")

# Install required packages for Drive backup
if ENABLE_DRIVE_BACKUP:
    print("\n📦 Installing Google Drive backup dependencies...")
    !pip install google-api-python-client google-auth google-auth-oauthlib google-auth-httplib2 -q
    print("✅ Dependencies installed")

In [ ]:
# Cell 3: Google Drive Authentication
import os
import json

print("=" * 60)
print("GOOGLE DRIVE AUTHENTICATION")
print("=" * 60)

# Import the backup module
import sys
sys.path.insert(0, '/kaggle/working')

# Check for credentials (supports both naming conventions)
# Uses same credential names as your n8n setup for consistency
client_id = os.environ.get('GDRIVE_CLIENT_ID') or os.environ.get('GOOGLE_CLIENT_ID') or os.environ.get('GOOGLE_OAUTH_CLIENT_ID', '')
client_secret = os.environ.get('GDRIVE_CLIENT_SECRET') or os.environ.get('GOOGLE_CLIENT_SECRET') or os.environ.get('GOOGLE_OAUTH_CLIENT_SECRET', '')

if ENABLE_DRIVE_BACKUP:
    print("\n📋 Google Drive Setup Requirements:")
    print("   Same credentials as your n8n setup (if you have them):")
    print("   - GDRIVE_CLIENT_ID / GDRIVE_CLIENT_SECRET")
    print("   - Or: GOOGLE_CLIENT_ID / GOOGLE_CLIENT_SECRET")
    print("   - Or: GOOGLE_OAUTH_CLIENT_ID / GOOGLE_OAUTH_CLIENT_SECRET")
    print("\n   If starting fresh:")
    print("   1. Create Google Cloud project at https://console.cloud.google.com")
    print("   2. Enable Google Drive API")
    print("   3. Create OAuth 2.0 credentials (Desktop app type)")
    print("   4. Add credentials to Kaggle Secrets:")
    print("      - GDRIVE_CLIENT_ID (or GOOGLE_CLIENT_ID)")
    print("      - GDRIVE_CLIENT_SECRET (or GOOGLE_CLIENT_SECRET)")
    print("   5. Run this cell to authenticate")
    
    if client_id and client_secret:
        print("\n✅ Credentials found in Kaggle Secrets")
        print(f"   Using: {os.environ.get('GDRIVE_CLIENT_ID') and 'GDRIVE_*' or os.environ.get('GOOGLE_OAUTH_CLIENT_ID') and 'GOOGLE_OAUTH_*' or 'GOOGLE_CLIENT_*'}")
        print("   Attempting authentication...")
        
        # Import and run authentication
        try:
            from drive_backup import GoogleDriveBackup
            drive = GoogleDriveBackup(dry_run=DRY_RUN)
            auth_success = drive.authenticate()
            
            if auth_success:
                print("\n✅ Google Drive authenticated successfully!")
                print("   You can now use backup/restore functions.")
            else:
                print("\n⚠️  Authentication failed or not completed")
                print("   See KAGGLE_MINECRAFT_GUIDE.md for setup help")
        except ImportError:
            print("⚠️  drive_backup.py not found")
            print("   Make sure the file is in the same directory as this notebook")
        except Exception as e:
            print(f"❌ Authentication error: {e}")
    else:
        print("\n⚠️  No Google credentials found in Kaggle Secrets")
        print("   Add credentials (see above for naming options)")
        print("   See guide for setup instructions")
else:
    print("\n⏭️  Google Drive backup disabled")
    print("   Set ENABLE_DRIVE_BACKUP=True to enable")

In [ ]:
# Cell 4: Drive Connection Test & Backup Folder Setup
if ENABLE_DRIVE_BACKUP:
    print("=" * 60)
    print("TESTING GOOGLE DRIVE CONNECTION")
    print("=" * 60)
    
    try:
        from drive_backup import GoogleDriveBackup
        
        drive = GoogleDriveBackup(dry_run=DRY_RUN)
        
        # Test authentication if not already done
        if not drive.authenticated:
            drive.authenticate()
        
        # Test connection
        if drive.test_connection():
            print("\n✅ Google Drive connection working!")
            
            # List existing backups
            backups = drive.list_backups()
            if backups:
                print(f"\n📋 Found {len(backups)} existing backup(s)")
            else:
                print("\n📁 No existing backups found")
                print("   First backup will be created after server starts")
        else:
            print("\n⚠️  Drive connection test failed")
            print("   Backups will be saved locally only")
            
    except Exception as e:
        print(f"❌ Error: {e}")
        print("   Continuing without Drive backup")
else:
    print("=" * 60)
    print("GOOGLE DRIVE CONNECTION SKIPPED")
    print("=" * 60)
    print("\n   Enable ENABLE_DRIVE_BACKUP to use Drive persistence")

In [ ]:
# Cell 5: World Restoration from Google Drive (Run Before Server Start)
import os
import sys

print("=" * 60)
print("WORLD RESTORATION FROM GOOGLE DRIVE")
print("=" * 60)

SERVER_DIR = "/kaggle/working/minecraft_server"

if ENABLE_DRIVE_BACKUP:
    try:
        from drive_backup import GoogleDriveBackup
        
        drive = GoogleDriveBackup(dry_run=DRY_RUN)
        
        if drive.authenticated:
            # Check if we have existing world
            existing_world = os.path.join(SERVER_DIR, "world")
            world_exists = os.path.isdir(existing_world)
            
            if world_exists:
                print("\n⚠️  Existing world detected!")
                print("   World will be preserved. To restore from backup:")
                print("   1. Set RESTORE_FROM_DRIVE = True below")
                print("   2. Set FORCE_REPLACE = True to overwrite")
                print("   3. Run this cell again")
                
                RESTORE_FROM_DRIVE = False  # Set True to restore
                FORCE_REPLACE = False  # Set True to overwrite without confirmation
                
                if RESTORE_FROM_DRIVE:
                    success = drive.restore_backup(force_replace=FORCE_REPLACE)
                    if success:
                        print("\n✅ World restored from Google Drive!")
                else:
                    print("\n✅ Keeping existing world")
            else:
                print("\n🔍 No existing world found")
                print("   Attempting to restore from Google Drive...")
                
                success = drive.restore_backup()
                
                if success:
                    print("\n✅ World restored from Google Drive!")
                else:
                    print("\nℹ️  No backup to restore or restoration failed")
                    print("   A new world will be created when server starts")
        else:
            print("\n⚠️  Not authenticated with Google Drive")
            print("   Run Cell 3 (Authentication) first")
    else:
        print("\n⚠️  Drive backup not enabled")
else:
    print("\n⏭️  Google Drive restoration skipped (disabled)")

In [ ]:
# Cell 6: Server Configuration
import os

SERVER_DIR = "/kaggle/working/minecraft_server"
EULA_ACCEPTED = True  # From Cell 2

print("=" * 60)
print("GENERATING SERVER CONFIGURATION")
print("=" * 60)

# EULA file
eula_path = os.path.join(SERVER_DIR, "eula.txt")
if EULA_ACCEPTED:
    with open(eula_path, 'w') as f:
        f.write("#By changing the setting below to TRUE you are indicating your agreement to our EULA.\n")
        f.write("#You must reboot the server after changing this option.\n")
        f.write("eula=true\n")
    print("✅ EULA accepted (eula=true)")
    print("   Remember: Run 'stop' in-game to shutdown gracefully")
else:
    print("⚠️  EULA NOT accepted. Server will not start!")
    print("   Go to https://minecraft.net/eula and agree, then set EULA_ACCEPTED=True")

# server.properties for Java Edition
props_path = os.path.join(SERVER_DIR, "server.properties")
with open(props_path, 'w') as f:
    f.write("""#Minecraft server properties
server-name=Kaggle Minecraft Server\ngamemode=survival\ndifficulty=normal\nlevel-type=default\nenable-command-block=false\nenable-query=false\nop-permission-level=4\nallow-nether=true\ndifficulty=normal\nhardcore=false\nenable-command-block=false\nmax-players=5\nnetwork-compression-threshold=256\nonline-mode=true\nplayer-idle-timeout=0\nmax-world-size=29999984\nspawn-npcs=true\nspawn-animals=true\nspawn-monsters=true\nspawn-protection=16\nview-distance=6\nsimulation-distance=4\nentity-broadcast-range-percentage=100\nseed=\nmotd=Kaggle Minecraft Server\ndebug=false\nprevent-proxy-connections=false\ngenerate-structures=true\nspawn-protection=16\nspawn-complexity=10\nmax-entities-cramming=100\nmax-weapon-damage=256\nmax-tick-time=60000\nmax-build-height=256\nnetwork-compression-threshold=256\nserver-ip=\nserver-port=25565\nsnooper-enabled=true\nwhite-list=false\nmax-world-size=29999984\nuse-native-transport=true\nverbose=true\nenable-rcon=false\nrcon.password=\nrcon.port=25575\n""")
print("✅ server.properties created")

print("\n📋 Configuration Summary:")
print("  - Gamemode: survival")
  - Difficulty: normal")
  - Max Players: 5")
  - View Distance: 6")
  - Online Mode: true")

print("\n" + "=" * 60)

In [ ]:
# Cell 7: Server Startup with Background Backup
import os
import subprocess
import time
import sys
import threading

SERVER_DIR = "/kaggle/working/minecraft_server"
JAVA_XMS = 1024
JAVA_XMX = 2048

print("=" * 60)
print("STARTING MINECRAFT SERVER")
print("=" * 60)

server_jar = os.path.join(SERVER_DIR, "paper.jar")

if not os.path.exists(server_jar):
    print("❌ Server jar not found! Run Cell 3 first.")
else:
    print(f"📦 Starting server from: {server_jar}")
    print(f"💾 Memory: {JAVA_XMS}M initial, {JAVA_XMX}M max")
    print("\n⏳ Waiting for server to start...")
    print("   (This may take 1-3 minutes for world generation)\n")
    
    # Start server
    cmd = f'java -Xms{JAVA_XMS}m -Xmx{JAVA_XMX}m -jar {server_jar} nogui'
    
    try:
        server_process = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            cwd=SERVER_DIR
        )
        
        print(f"✅ Server process started (PID: {server_process.pid})")
        
        # Setup background backup if Drive enabled
        if ENABLE_DRIVE_BACKUP:
            print("\n🔄 Setting up periodic Google Drive backups...")
            print(f"   Backup interval: {BACKUP_INTERVAL_MINUTES} minutes")
            print("   ⚠️  Note: Kaggle may terminate sessions without warning")
            print("   ⚠️  Don't rely solely on periodic backups")
            
            # Store reference for cleanup
            backup_thread = None
            
            def periodic_backup():
                """Run backups periodically in background"""
                from drive_backup import GoogleDriveBackup
                
                while server_process.poll() is None:
                    try:
                        drive = GoogleDriveBackup(dry_run=DRY_RUN)
                        if drive.authenticated:
                            drive.create_backup()
                    except Exception as e:
                        print(f"Backup error: {e}")
                    
                    # Wait for next interval
                    for _ in range(BACKUP_INTERVAL_MINUTES * 60):
                        if server_process.poll() is not None:
                            break
                        time.sleep(1)
            
            # Start backup thread
            backup_thread = threading.Thread(target=periodic_backup, daemon=True)
            backup_thread.start()
            print("✅ Background backup thread started")
        
        print("\n📋 To check server status, run the health check cell below")
        print("📋 To stop the server, press the stop button in the notebook toolbar")
        
    except Exception as e:
        print(f"❌ Failed to start server: {e}")

print("\n" + "=" * 60)

In [ ]:
# Cell 8: Playit.gg Tunnel Setup
print("=" * 60)
print("PLAYIT.GG TUNNEL SETUP")
print("=" * 60)

print("""
Playit.gg provides free tunneling for game servers without port forwarding.

✅ SUPPORTED:
   - Java Edition TCP connections (port 25565)
   - Persistent hostname/port (doesn't change every session)
   - Free tier available

⚠️  BEDROCK LIMITATIONS:
   - Geyser plugin does NOT work with Playit UDP tunnels
   - Bedrock 1.26.51+ uses NetherNet (may need paid plan)
   - Free tier Bedrock support is limited/unreliable
   - For crossplay, may need alternative solution

📋 SETUP STEPS:
   1. Create account at https://playit.gg
   2. Download agent or use our automatic downloader
   3. Claim your server via dashboard
   4. Get your token and add to Kaggle Secrets
   5. Run this cell to start the tunnel
""")

import os
import sys
sys.path.insert(0, '/kaggle/working')

from playit_setup import PlayitTunnel

print("\n🔧 Initializing Playit tunnel...")

# Initialize tunnel
tunnel = PlayitTunnel(dry_run=False)

# Setup directory
tunnel.setup_directory()

# Check for existing token
token = tunnel._get_token()
if token:
    print("✅ Token found in Kaggle Secrets (PLAYIT_TOKEN)")
    print("   Starting agent with existing token...")
else:
    print("⚠️  No PLAYIT_TOKEN found in Kaggle Secrets")
    print("\n📋 To get a token:")
    print("   1. Visit https://playit.gg and sign up")
    print("   2. Download & run Playit agent locally first")
    print("   3. Claim your server in the dashboard")
    print("   4. Copy your account token")
    print("   5. Add PLAYIT_TOKEN to Kaggle Secrets")
    print("\n💡 Or run the agent now and claim via dashboard:")
    print("   The agent will show a claim code to enter at playit.gg")

# Download agent (skip if you uploaded manually)
print("\n📥 Downloading Playit agent...")
binary_path = tunnel.download_agent()

if binary_path:
    # Start agent
    print("\n🚀 Starting Playit agent...")
    if tunnel.start_agent(binary_path):
        print("\n⏳ Waiting for agent to be ready...")
        time.sleep(30)
        
        # Get public address
        addr = tunnel.get_public_address()
        if addr:
            hostname, port = addr
            print("\n✅ PLAYIT TUNNEL ACTIVE")
            print(f"📍 Public hostname: {hostname}")
            print(f"📍 Public port: {port}")
            print(f"🔗 Connection: {hostname}:{port}")
            print("\n💡 Share this address with Java Edition players")
        else:
            print("\n⚠️  Agent started but address not detected")
            print("   Check https://playit.gg/dashboard for your address")
    else:
        print("❌ Failed to start agent")
else:
    print("⚠️  Agent binary not available")
    print("   Upload manually from https://playit.gg/download")
    print("   Or run tunnel later via dashboard")

print("\n" + "=" * 60)

In [ ]:
# Cell 9: Connection Information and Health Checks
import os
import time

SERVER_DIR = "/kaggle/working/minecraft_server"

print("=" * 60)
print("SERVER STATUS & CONNECTION INFO")
print("=" * 60)

# Check if server is running
server_properties = os.path.join(SERVER_DIR, "server.properties")

print("\n📋 Server Configuration:")
print(f"   Directory: {SERVER_DIR}")
print(f"   Port: 25565 (Java Edition default)")

if os.path.exists(server_properties):
    with open(server_properties, 'r') as f:
        props = f.read()
    
    # Parse key settings
    for line in props.split('\n'):
        if '=' in line and not line.startswith('#'):
            key, val = line.split('=', 1)
            if key.strip() in ['max-players', 'view-distance', 'difficulty', 'online-mode', 'motd']:
                print(f"   {key.strip()}: {val.strip()}")

print("\n📋 How Friends Connect - JAVA EDITION:")
print("   Using Playit.gg tunnel (from Cell 8):")
print("   1. Open Minecraft → Multiplayer → Add Server")
print("   2. Server Address: (Playit hostname from Cell 8)")
print("   3. Server Port: (Playit port from Cell 8)")
print("   OR: Direct connection link")

print("\n📋 How Friends Connect - BEDROCK EDITION:")
print("   ⚠️  Bedrock support via Playit is LIMITED on free tier")
print("   - Geyser plugin does NOT work with Playit UDP")
print("   - Try: Playit dashboard → Bedrock tunnel (if available)")
print("   - If not available: Ask friends to use Java Edition")
print("   - Alternative: Use separate hosting for Bedrock")

print("\n📋 Server Logs (last 20 lines):")
# Show recent log entries
log_file = os.path.join(SERVER_DIR, "logs", "latest.log")
if os.path.exists(log_file):
    with open(log_file, 'r') as f:
        lines = f.readlines()
    for line in lines[-20:]:
        print(f"   {line.rstrip()}")
else:
    print("   (No logs yet - server may still be starting)")

print("\n" + "=" * 60)

In [ ]:
# Cell 10: Google Drive Backup Status & Manual Operations
import os
import sys

print("=" * 60)
print("GOOGLE DRIVE BACKUP STATUS & OPERATIONS")
print("=" * 60)

if ENABLE_DRIVE_BACKUP:
    try:
        from drive_backup import GoogleDriveBackup, test_backup_system
        
        drive = GoogleDriveBackup(dry_run=DRY_RUN)
        
        if not drive.authenticated:
            drive.authenticate()
        
        # Show backup status
        print("\n📊 BACKUP STATUS:")
        backups = drive.list_backups()
        
        print("\n🛠️  AVAILABLE OPERATIONS:")
        print("   To create a manual backup now, run:")
        print("   ```python")
        print("   drive = GoogleDriveBackup()")
        print("   drive.create_backup()")
        print("   ```")
        print("\n   To restore a specific backup, run:")
        print("   ```python")
        print("   drive = GoogleDriveBackup()")
        print("   # List backups first to get ID")
        print("   drive.restore_backup(backup_id='YOUR_BACKUP_ID')")
        print("   ```")
        
    except Exception as e:
        print(f"❌ Error: {e}")
else:
    print("\n⏭️  Google Drive backup disabled")

print("\n" + "=" * 60)

In [ ]:
# Cell 11: Logs, Troubleshooting, and Shutdown
import os
import shutil
import zipfile
from datetime import datetime

SERVER_DIR = "/kaggle/working/minecraft_server"
BACKUP_DIR = "/kaggle/working/backups"
os.makedirs(BACKUP_DIR, exist_ok=True)

print("=" * 60)
print("LOGS, BACKUP & SHUTDOWN HANDLING")
print("=" * 60)

# ===== GOOGLE DRIVE BACKUP (if enabled) =====
if ENABLE_DRIVE_BACKUP:
    print("\n☁️  GOOGLE DRIVE BACKUP:")
    print("   Your world is being backed up to Google Drive automatically.")
    print("   To create a final backup before closing:")
    print("```python")
    print("from drive_backup import GoogleDriveBackup")
    print("drive = GoogleDriveBackup()")
    print("drive.create_backup()  # Final backup before shutdown")
    print("```")

# ===== LOCAL BACKUP (fallback) =====
print("\n💾 LOCAL BACKUP (fallback):")
print("   If Drive backup fails, you can backup locally:")

world_dir = os.path.join(SERVER_DIR, "world")
if os.path.exists(world_dir):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_name = f"world_backup_{timestamp}.zip"
    backup_path = os.path.join(BACKUP_DIR, backup_name)
    
    print(f"   Create local backup: {backup_name}")
    print(f"   Download from Kaggle Files panel after creation")
    print("```python")
    print("!zip -r /kaggle/working/backups/world_backup.zip /kaggle/working/minecraft_server/world/")
    print("```")
else:
    print("   No world found yet")

# ===== RESTORATION INSTRUCTIONS =====
print("\n🔄 RESTORING FROM GOOGLE DRIVE:")
print("   1. Authenticate with Drive (Cell 3)")
   2. List backups: drive.list_backups()")
   3. Restore: drive.restore_backup()")
   4. Start server (Cell 7)")

# ===== COMMON ISSUES =====
print("\n🛠️  TROUBLESHOOTING:")
print("""
❌ Drive backup not working:
   - Check credentials in Kaggle Secrets (any of these naming conventions):
     - GDRIVE_CLIENT_ID / GDRIVE_CLIENT_SECRET (n8n style)
     - GOOGLE_OAUTH_CLIENT_ID / GOOGLE_OAUTH_CLIENT_SECRET (n8n Google services)
     - GOOGLE_CLIENT_ID / GOOGLE_CLIENT_SECRET (generic)
   - Verify OAuth consent screen is configured in Google Cloud Console
   - Re-run authentication cell

❌ Server won't start:
   - Check Java is installed: !apt-get update && !apt-get install -y default-jdk
   - Verify EULA was accepted (eula.txt should contain 'eula=true')
   - Check available RAM - reduce JAVA_XMX if OOM errors

❌ Can't connect from outside:
   - Make sure Playit agent is running (Cell 8)
   - Check Playit dashboard for your address
   - Verify PLAYIT_TOKEN is set correctly

❌ Lag/Performance issues:
   - Reduce VIEW_DISTANCE to 4-6
   - Reduce MAX_PLAYERS
   - Use Paper server (already configured) for better performance

❌ World not saving:
   - Always run 'stop' command in-game before closing
   - Run final Drive backup before closing notebook
""")

# ===== GRACEFUL SHUTDOWN =====
print("\n🛑 SHUTDOWN PROCEDURE:")
print("   1. Type 'save-all' in server console to flush world")
   2. Type 'stop' to shutdown server gracefully")
   3. Run final Drive backup (see above)")
   4. Stop the notebook kernel")

print("\n⚠️  IMPORTANT WARNINGS:")
print("   - Kaggle may terminate sessions without warning")
   - Periodic backups help but are not guaranteed")
   - Always do a manual backup before important sessions")
   - World data lost if session ends without backup")

print("\n" + "=" * 60)
print("SETUP COMPLETE!")
print("=" * 60)
print("\n📋 NEXT STEPS:")
print("   1. Run Cell 7 to start the server")
   2. Wait for world generation (1-3 min)")
   3. Run Cell 8 to start Playit tunnel")
   4. Share the Playit address with friends")
   5. Periodic Drive backups will run automatically")
   6. Before closing: final backup + graceful shutdown")
print("\n🎮 Enjoy your Kaggle Minecraft server!")
print("=" * 60)

In [ ]:
# Cell 12: RaheeCraft Control Agent (power controls, console, monitoring)
#
# Connects this Kaggle session to your deployed control panel backend.
# The agent ONLY makes outbound HTTPS calls (register, heartbeat, poll
# actions, stream logs) — Kaggle never accepts inbound connections.
#
# Required — set these BEFORE running (do NOT hardcode secrets here):
#   In Kaggle: Add-ons -> Secrets -> add BACKEND_URL, AGENT_ID, AGENT_SECRET
#   (BACKEND_URL = your Render URL, e.g. https://raheecraft-control-panel.onrender.com)

import os, sys, subprocess, threading

# --- 1. Read credentials from Kaggle secrets / environment (never hardcoded) ---
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _key in ("BACKEND_URL", "AGENT_ID", "AGENT_SECRET"):
        if not os.environ.get(_key):
            try:
                os.environ[_key] = _secrets.get_secret(_key)
            except Exception:
                pass
except Exception:
    pass

missing = [k for k in ("BACKEND_URL", "AGENT_ID", "AGENT_SECRET") if not os.environ.get(k)]
if missing:
    print(f"❌ Missing secrets: {missing}")
    print("   Add them via Kaggle Secrets (Add-ons -> Secrets) or os.environ, then rerun.")
    print("   Generate agent credentials first:  python agent/register_agent.py")
else:
    # --- 2. Fetch the agent code (clone the repo, or it may already be uploaded) ---
    if not os.path.exists("/kaggle/working/rahee-world/agent/main.py"):
        try:
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/soldiers1234583/raheecraft-control-panel.git",
                 "/kaggle/working/rahee-world"],
                check=True, capture_output=True, timeout=120,
            )
            print("✅ Repo cloned")
        except Exception as e:
            print(f"⚠️  Clone failed ({e}). Upload the repo's backend/ and agent/ folders")
            print("   to /kaggle/working/rahee-world/ manually, then rerun this cell.")

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx"], check=False)

    sys.path.insert(0, "/kaggle/working/rahee-world")
    os.environ.setdefault("SERVER_DIR", "/kaggle/working/minecraft_server")

    # --- 3. Run the agent in a background thread (heartbeats + action polling) ---
    from agent.main import run_agent

    def _agent_loop():
        try:
            run_agent()
        except Exception as e:
            print(f"Agent stopped: {e}")

    _agent_thread = threading.Thread(target=_agent_loop, daemon=True, name="rahee-agent")
    _agent_thread.start()

    print("✅ Control agent started (background thread)")
    print(f"   Backend: {os.environ['BACKEND_URL']}")
    print(f"   Agent ID: {os.environ['AGENT_ID'][:8]}... (redacted)")
    print("   Watch heartbeats in the dashboard, or the backend's Render logs.")